# Low-Rank Signature Features

Signature features via an iterative low-rank approximation of the tensor outer products ([Király & Oberhauser, JMLR 2019](https://jmlr.org/papers/volume20/16-314/16-314.pdf), Alg. 5): a random projection of the signature, with **no** Random Fourier static features. `P @ P.T` approximates the full-rank Gram.

## Environment
Detect the live backend/device and whether a SYCL fast-path is available.

In [ ]:
import sys, pathlib
# Make `_nbtools` and the in-repo `ksig` importable whether the notebook is
# launched from ./notebooks or from the repo root (no `pip install` needed).
_nbdir = pathlib.Path.cwd()
_root = _nbdir.parent if (_nbdir / "_nbtools.py").exists() else _nbdir
_nbdir = _root / "notebooks"
for _p in (str(_nbdir), str(_root)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import ksig
import _nbtools as nb
%matplotlib inline

ENV = nb.detect_env()
nb.print_env_banner(ENV)

## Deterministic input
`simulate(24, 20, 3, seed=0)` — integrated random walks (the NumPy RNG is identical on every machine).

In [ ]:
X = nb.simulate(24, 20, 3, seed=0)
print('X shape:', X.shape, '| dtype:', X.dtype)

## Compute the feature map

**Reference output — NVIDIA H100 NVL (CuPy 14.1.0), kwargs: `n_levels=4, TRP(n_components=100, rank=1), normalize=True`:**

```text
gram shape   : (24, 24)
feature shape: (24, 401)
diag mean    : 1.0
rel err vs exact: 0.3997
```

> ⚠️ **Portability:** these feature maps are **random**. `diag == 1` (from
> `normalize=True`) and the Gram's symmetry are exact on any backend, but the
> *element values* and the exact `rel err` depend on the RNG stream, which
> differs between CuPy and torch. Expect the **same ballpark** rel-err, and the
> same **convergence** — it shrinks as `n_components` grows:
> `50:0.4242  100:0.3997  250:0.3811  500:0.3795` (n_components:rel_err, on NVIDIA H100 NVL).

In [ ]:
N_COMPONENTS = 100                   # the value the reference was frozen at
k = ksig.kernels.SignatureFeatures(n_levels=4, order=1, normalize=True, static_features=None, projection=ksig.projections.TensorizedRandomProjection(n_components=N_COMPONENTS, rank=1, random_state=0))
k.fit(X)
K = nb.as_host(k(X)); P = nb.as_host(k.transform(X))
exact = ksig.kernels.SignatureKernel(n_levels=4, order=1, normalize=True,
                                     static_kernel=ksig.static.kernels.RBFKernel())
Kex = nb.as_host(exact(X))
rel = float(np.linalg.norm(K - Kex) / np.linalg.norm(Kex))
print("gram shape   :", tuple(K.shape))
print("feature shape:", tuple(P.shape))
print("diag mean    :", round(float(np.diag(K).mean()), 6))
print("rel err vs exact:", round(rel, 4))

## Scaling — green = CUDA reference, blue = this machine

The cell below sweeps **n_samples N  (L=50, d=5, n_components=100)** and times each point on whatever backend
is live, then overlays:

* 🟩 **green** — the frozen reference measured on **NVIDIA H100 NVL** (`cuda_reference.json`),
* 🟦 **blue** — what *this* machine computes now (torch-native on Aurora XPU / CUDA / CPU),
* 🟧 **orange** — the optional **SYCL** fast-path, drawn **only if** a `ksig._sycl`
  build is present (`nb.sycl_available()`); if SYCL is never adopted, no orange curve.

The grid and the knobs at the top of the cell are **tunable** — they default to
the reference grid so blue lines up with green; widen them to push the frontier.

In [ ]:
# --- tunable knobs (default to the CUDA-reference grid) ---------------------
N_GRID = [64, 128, 256, 512, 1024]     # sample counts to sweep (feature methods scale ~linearly)
L, D   = 50, 5                  # fixed sequence length / channels
REPS   = 5
N_COMPONENTS = 100                # projection width (rank of the approx)

def time_one(N):
    Xs = nb.simulate(N, L, D, seed=1)
    k = ksig.kernels.SignatureFeatures(n_levels=4, order=1, normalize=True, static_features=None, projection=ksig.projections.TensorizedRandomProjection(n_components=N_COMPONENTS, rank=1, random_state=0))
    k.fit(Xs)
    return nb.timeit(lambda: k(Xs), reps=REPS, device=ENV["device"])

times = [time_one(N) for N in N_GRID]

# Second (orange) curve ONLY when a SYCL build is available.
sycl_times = None
if nb.sycl_available():
    nb.enable_sycl(True)
    sycl_times = [time_one(N) for N in N_GRID]
    nb.enable_sycl(False)

nb.scaling_plot(N_GRID, times, "low_rank_signature", sycl_seconds=sycl_times,
                title="low_rank_signature — wall time vs n_samples");